In [7]:
!pip install -q pypdf pdfplumber sentence-transformers faiss-cpu google-generativeai matplotlib seaborn tqdm

In [36]:
import os
import re
import time
import textwrap
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm

import pypdf
import pdfplumber
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

All libraries imported successfully!


#API Key Setup

In [37]:
# Option 1: Direct entry (not recommended for sharing)
# GEMINI_API_KEY = "your-api-key-here"

# Option 2: Using Colab Secrets (recommended)
# Go to  Secrets in left panel → Add GEMINI_API_KEY
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
print('Gemini API configured!')

Gemini API configured!


#Upload the ML Book PDF

In [38]:
# from google.colab import files

# print(' Please upload the PDF book (Hands-On ML with Scikit-Learn, Keras & TF)...')
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]
# print(f' Uploaded: {PDF_PATH}')

# Or, uncomment the line below and paste your PDF path here directly!
PDF_PATH = '/content/intro-to-ml.pdf' # <--- PASTE YOUR PDF PATH HERE
print(f' Using PDF from path: {PDF_PATH}')

 Using PDF from path: /content/intro-to-ml.pdf


---
# Part 1 — Data Understanding & Preprocessing
##  1.1 Load & Explore the PDF

In [39]:
# ── Load with pypdf for metadata ──────────────────────────────────────────────
reader = pypdf.PdfReader(PDF_PATH)
total_pages = len(reader.pages)

print('=' * 60)
print('DOCUMENT STRUCTURE EXPLORATION')
print('=' * 60)
print(f'  Total Pages   : {total_pages}')

# Metadata
meta = reader.metadata
print(f'  Title         : {meta.title or "N/A"}')
print(f'  Author        : {meta.author or "N/A"}')
print(f'  Creator       : {meta.creator or "N/A"}')
print(f'  Producer      : {meta.producer or "N/A"}')

DOCUMENT STRUCTURE EXPLORATION
  Total Pages   : 392
  Title         : Introduction to Machine Learning with Python
  Author        : Andreas C. Müller and Sarah Guido
  Creator       : AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)
  Producer      : 3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)


In [40]:
# ── Extract text from ALL pages using pdfplumber ───────────────────────────────
print('Extracting text from all pages...')
raw_pages = []   # list of (page_num, text)

with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(tqdm(pdf.pages, desc='Extracting')):
        text = page.extract_text() or ''
        raw_pages.append({'page': i + 1, 'text': text, 'char_count': len(text)})

total_chars = sum(p['char_count'] for p in raw_pages)
non_empty   = sum(1 for p in raw_pages if p['char_count'] > 50)

print(f'\n Extraction complete!')
print(f'   Non-empty pages : {non_empty} / {total_pages}')
print(f'   Total characters: {total_chars:,}')
print(f'   Avg chars/page  : {total_chars // total_pages:,}')

Extracting text from all pages...


Extracting: 100%|██████████| 392/392 [00:29<00:00, 13.34it/s]


 Extraction complete!
   Non-empty pages : 387 / 392
   Total characters: 669,775
   Avg chars/page  : 1,708


In [41]:
# ── Text quality analysis ──────────────────────────────────────────────────────
print('TEXT QUALITY ANALYSIS')
print('-' * 40)

sample_issues = []
for p in raw_pages[:50]:
    txt = p['text']
    if re.search(r'\x00|\ufffd|\u200b', txt):   # Null bytes / replacement chars
        sample_issues.append(('encoding_issue', p['page']))
    if re.search(r'\f', txt):                    # Form feeds
        sample_issues.append(('form_feed', p['page']))

empty_pages   = [p['page'] for p in raw_pages if p['char_count'] < 50]
sparse_pages  = [p['page'] for p in raw_pages if 50 <= p['char_count'] < 200]

print(f'  Empty/near-empty pages  : {len(empty_pages)}')
print(f'  Sparse pages (<200 ch)  : {len(sparse_pages)}')
print(f'  Encoding issues found   : {len([i for i in sample_issues if i[0]=="encoding_issue"])}')
print(f'\n  Sample empty page nums  : {empty_pages[:10]}')

# Show sample text
sample_page = next((p for p in raw_pages if p['char_count'] > 1000), raw_pages[10])
print(f'\nSample text from page {sample_page["page"]}:')
print('-' * 50)
print(sample_page['text'][:600])

TEXT QUALITY ANALYSIS
----------------------------------------
  Empty/near-empty pages  : 5
  Sparse pages (<200 ch)  : 6
  Encoding issues found   : 0

  Sample empty page nums  : [2, 144, 224, 318, 336]

Sample text from page 4:
--------------------------------------------------
Introduction to Machine Learning with Python
by Andreas C. Müller and Sarah Guido
Copyright © 2017 Sarah Guido, Andreas Müller. All rights reserved.
Printed in the United States of America.
Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles (http://safaribooksonline.com). For more information, contact our corporate/
institutional sales department: 800-998-9938 or corporate@oreilly.com.
Editor: Dawn Schanafelt Indexer: Judy


# 1.3 Text Cleaning & Chunking

In [42]:
def clean_text(text: str) -> str:
    """Clean extracted PDF text."""
    text = re.sub(r'\x00|\ufffd|\u200b', '', text)    # Remove null/garbage chars
    text = re.sub(r'\f', '\n', text)                   # Form feed → newline
    text = re.sub(r'-\n(\w)', r'\1', text)             # Rejoin hyphenated words
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)       # Single newlines → space
    text = re.sub(r'\n{3,}', '\n\n', text)             # Multiple blanks → double
    text = re.sub(r' {2,}', ' ', text)                 # Multiple spaces
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)        # Non-ASCII
    return text.strip()


def chunk_text(text: str, page_num: int,
               chunk_size: int = 800,
               overlap: int = 100) -> list:
    """
    Split text into overlapping chunks.
    Tries to split at sentence boundaries first.
    """
    words = text.split()
    if len(words) == 0:
        return []

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_words = words[start:end]
        chunk_text_str = ' '.join(chunk_words)

        # Try to end at sentence boundary
        if end < len(words):
            last_period = chunk_text_str.rfind('. ')
            if last_period > chunk_size * 3:   # at least 3/4 through
                chunk_text_str = chunk_text_str[:last_period + 1]

        if len(chunk_text_str.strip()) > 50:
            chunks.append({
                'chunk_id'  : f'page{page_num}_chunk{chunk_id}',
                'page'      : page_num,
                'text'      : chunk_text_str.strip(),
                'word_count': len(chunk_words)
            })
            chunk_id += 1

        start = end - overlap
        if start >= len(words) - overlap:
            break

    return chunks


# ── Apply to all pages ─────────────────────────────────────────────────────────
CHUNK_SIZE = 800   # words
OVERLAP    = 100   # words

all_chunks = []
for page_info in tqdm(raw_pages, desc='Chunking'):
    cleaned = clean_text(page_info['text'])
    chunks  = chunk_text(cleaned, page_info['page'],
                         chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    all_chunks.extend(chunks)

print(f'\n Chunking complete!')
print(f'   Total chunks created : {len(all_chunks):,}')
print(f'   Chunk size setting   : {CHUNK_SIZE} words')
print(f'   Overlap setting      : {OVERLAP} words')

Chunking: 100%|██████████| 392/392 [00:00<00:00, 5857.92it/s]


 Chunking complete!
   Total chunks created : 387
   Chunk size setting   : 800 words
   Overlap setting      : 100 words


---
# Part 2 — Embedding & Vector Database
# 2.1 Generate Embeddings with SentenceTransformers

In [43]:
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

print(f'Loading SentenceTransformer: {EMBEDDING_MODEL}')
embedder = SentenceTransformer(EMBEDDING_MODEL)

print(f'\n Model Details:')
print(f'   Model name       : {EMBEDDING_MODEL}')
print(f'   Embedding dim    : {embedder.get_sentence_embedding_dimension()}')
print(f'   Max sequence len : {embedder.max_seq_length}')
print(f'   Desc             : Lightweight, fast, 384-dim embeddings ideal for semantic search')

Loading SentenceTransformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Model Details:
   Model name       : all-MiniLM-L6-v2
   Embedding dim    : 384
   Max sequence len : 256
   Desc             : Lightweight, fast, 384-dim embeddings ideal for semantic search


In [44]:
# ── Encode all chunks (batched for speed) ─────────────────────────────────────
texts = [c['text'] for c in all_chunks]

print(f' Encoding {len(texts):,} chunks in batches...')
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True    # L2-normalize for cosine similarity
)

print(f'\nEmbeddings generated!')
print(f'   Shape  : {embeddings.shape}')
print(f'   dtype  : {embeddings.dtype}')
print(f'   Memory : ~{embeddings.nbytes / 1024**2:.1f} MB')

 Encoding 387 chunks in batches...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embeddings generated!
   Shape  : (387, 384)
   dtype  : float32
   Memory : ~0.6 MB


# 2.2 Build FAISS Vector Index

In [45]:
EMBEDDING_DIM = embeddings.shape[1]

# We use IndexFlatIP (inner product = cosine sim since embeddings are normalised)
faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)

# Add all embeddings to the index
faiss_index.add(embeddings.astype(np.float32))

print('FAISS Index Built!')
print(f'   Index type     : IndexFlatIP (exact cosine similarity)')
print(f'   Embedding dim  : {EMBEDDING_DIM}')
print(f'   Vectors stored : {faiss_index.ntotal:,}')
print(f'   Memory usage   : ~{faiss_index.ntotal * EMBEDDING_DIM * 4 / 1024**2:.1f} MB')

# Save index for later reuse
faiss.write_index(faiss_index, 'ml_book.faiss')
print('\n Index saved to ml_book.faiss')

FAISS Index Built!
   Index type     : IndexFlatIP (exact cosine similarity)
   Embedding dim  : 384
   Vectors stored : 387
   Memory usage   : ~0.6 MB

 Index saved to ml_book.faiss


In [46]:
# ── Verify retrieval with a sample query ──────────────────────────────────────
def retrieve_chunks(query: str, k: int = 5, verbose: bool = True):
    """Retrieve top-k relevant chunks for a query."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(q_emb, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        chunk = all_chunks[idx]
        results.append({
            'rank'   : rank + 1,
            'score'  : float(score),
            'page'   : chunk['page'],
            'chunk_id': chunk['chunk_id'],
            'text'   : chunk['text']
        })
    return results


# Test
sample_query = 'What is gradient descent and how does it work?'
results = retrieve_chunks(sample_query, k=3)

print(f'Query: "{sample_query}"')
print('=' * 65)
for r in results:
    print(f'  Rank {r["rank"]} | Score: {r["score"]:.4f} | Page: {r["page"]}')
    print(f'  {r["text"][:250]}...')
    print('-' * 65)

Query: "What is gradient descent and how does it work?"
  Rank 1 | Score: 0.4302 | Page: 103
  The main idea behind gradient boosting is to combine many simple models (in this context known as weak learners), like shallow trees. Each tree can only provide good predictions on part of the data, and so more and more trees are added to iteratively...
-----------------------------------------------------------------
  Rank 2 | Score: 0.3415 | Page: 120
  Figure 2-45. Illustration of a multilayer perceptron with a single hidden layer This model has a lot more coefficients (also called weights) to learn: there is one between every input and every hidden unit (which make up the hidden layer), and one be...
-----------------------------------------------------------------
  Rank 3 | Score: 0.3110 | Page: 370
  and gensim provide functionality for the techniques discussed in this paper and its follow-ups. Another direction in NLP that has picked up momentum in recent years is the use of recurren

---
# Part 3 — Retrieval Pipeline
#3.1 Query Embedding & Top-k Retrieval

In [48]:
def display_retrieval(query: str, k: int = 5):
    """Pretty-print retrieval results."""
    print(f'\n QUERY  : "{query}"')
    print(f'   Top-k  : {k}')
    print('=' * 70)
    results = retrieve_chunks(query, k=k)
    for r in results:
        print(f'  [{r["rank"]}] Score={r["score"]:.4f} | Page {r["page"]} | {r["chunk_id"]}')
        wrapped = textwrap.fill(r['text'][:300] + '...', width=68, initial_indent='     ', subsequent_indent='     ')
        print(wrapped)
        print()
    return results


test_queries = [
    'Explain the backpropagation algorithm in neural networks',
    'What are support vector machines?',
    'How does the random forest algorithm work?'
]

for q in test_queries:
    display_retrieval(q, k=3)


 QUERY  : "Explain the backpropagation algorithm in neural networks"
   Top-k  : 3
  [1] Score=0.4592 | Page 120 | page120_chunk0
     Figure 2-45. Illustration of a multilayer perceptron with a
     single hidden layer This model has a lot more coefficients
     (also called weights) to learn: there is one between every
     input and every hidden unit (which make up the hidden layer),
     and one between every unit in the hidden layer and the outpu...

  [2] Score=0.3636 | Page 122 | page122_chunk0
     In[92]: mglearn.plots.plot_two_hidden_layer_graph() Figure
     2-47. A multilayer perceptron with two hidden layers Having
     large neural networks made up of many of these layers of
     computation is what inspired the term deep learning. Tuning
     neural networks Let s look into the workings of the MLP by
     apply...

  [3] Score=0.3519 | Page 10 | page10_chunk0
     how to use the large array of models already implemented in
     scikit-learn and other libraries. Why We W

---
# Part 4 — Answer Generation (RAG) with Gemini 2.5 Flash
## 4.1 Setup Gemini LLM

In [50]:
# Initialize Gemini 2.5 Flash
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

# Verify connection
test_resp = gemini_model.generate_content('Say "RAG system ready!" and nothing else.')
print(f'Gemini 2.5 Flash: {test_resp.text.strip()}')

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 45.830994253s.

# 4.2 Prompt Design

In [26]:
SYSTEM_PROMPT = """\
You are an expert Machine Learning tutor. Your knowledge comes EXCLUSIVELY from
the provided book excerpts below. Follow these rules strictly:

1. GROUNDING: Answer only using information present in the provided context.
2. CITATION: When referencing content, mention the page number (e.g., "According to
   page X...").
3. HONESTY: If the context does not contain enough information to answer the
   question, say: "The provided excerpts do not contain enough information to
   answer this question fully."
4. CLARITY: Structure your answer clearly with explanations suitable for an ML
   student.
5. NO HALLUCINATION: Do NOT use any external knowledge beyond the provided
   context.
"""

def build_rag_prompt(query: str, chunks: list) -> str:
    """Construct the full RAG prompt with retrieved context."""
    context_blocks = []
    for i, chunk in enumerate(chunks):
        context_blocks.append(
            f'--- Excerpt {i+1} | Page {chunk["page"]} ---\n{chunk["text"]}'
        )
    context_str = '\n\n'.join(context_blocks)

    prompt = f"""{SYSTEM_PROMPT}

=== BOOK EXCERPTS ===
{context_str}
=== END OF EXCERPTS ===

STUDENT QUESTION: {query}

ANSWER (based only on the excerpts above):"""
    return prompt


print(' RAG Prompt template designed!')
print('\n Prompt Design Principles:')
print('  1. Strict grounding — model must not use outside knowledge')
print('  2. Citation requirement — forces use of page numbers')
print('  3. Fallback clause — prevents fabrication when context is sparse')
print('  4. Role definition — sets expert tutor persona for clarity')

✅ RAG Prompt template designed!

📋 Prompt Design Principles:
  1. Strict grounding — model must not use outside knowledge
  2. Citation requirement — forces use of page numbers
  3. Fallback clause — prevents fabrication when context is sparse
  4. Role definition — sets expert tutor persona for clarity


# 4.3 Full RAG Pipeline Function

In [27]:
def rag_answer(query: str, k: int = 5,
               temperature: float = 0.2,
               show_context: bool = False) -> dict:
    """
    Full RAG pipeline:
      1. Embed query
      2. Retrieve top-k chunks from FAISS
      3. Build grounded prompt
      4. Generate answer with Gemini 2.5 Flash
    """
    # Step 1 & 2: Retrieve
    chunks = retrieve_chunks(query, k=k)

    # Step 3: Build prompt
    prompt = build_rag_prompt(query, chunks)

    # Step 4: Generate
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=temperature,
            max_output_tokens=1024,
        )
    )
    answer = response.text.strip()

    result = {
        'query'   : query,
        'answer'  : answer,
        'k'       : k,
        'chunks'  : chunks,
        'sources' : sorted({c['page'] for c in chunks})
    }

    # Display
    print('\n' + '═' * 70)
    print(f' QUESTION: {query}')
    print('─' * 70)
    if show_context:
        print(' RETRIEVED CONTEXT:')
        for c in chunks:
            print(f'   [Page {c["page"]} | Score {c["score"]:.3f}]: {c["text"][:150]}...')
        print('─' * 70)
    print(f' ANSWER (Gemini 2.5 Flash | k={k} | pages {result["sources"]}):')
    print()
    for line in answer.split('\n'):
        print(textwrap.fill(line, width=70, subsequent_indent='   '))
    print('═' * 70)

    return result


print(' RAG pipeline ready!')

✅ RAG pipeline ready!


##  4.4 Sample Queries & Generated Answers

In [28]:
# Query 1
r1 = rag_answer(
    query='What is gradient descent and how does it minimize the loss function?',
    k=5,
    show_context=True
)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1786.64ms



══════════════════════════════════════════════════════════════════════
❓ QUESTION: What is gradient descent and how does it minimize the loss function?
──────────────────────────────────────────────────────────────────────
📄 RETRIEVED CONTEXT:
   [Page 103 | Score 0.395]: The main idea behind gradient boosting is to combine many simple models (in this context known as weak learners), like shallow trees. Each tree can on...
   [Page 120 | Score 0.333]: Figure 2-45. Illustration of a multilayer perceptron with a single hidden layer This model has a lot more coefficients (also called weights) to learn:...
   [Page 63 | Score 0.324]: Out[30]: Training set score: 0.95 Test set score: 0.61 This discrepancy between performance on the training set and the test set is a clear sign of ov...
   [Page 39 | Score 0.291]: CHAPTER 2 Supervised Learning As we mentioned earlier, supervised machine learning is one of the most commonly used and successful types of machine le...
   [Page 139 | Score 0.28

In [29]:
# Query 2
r2 = rag_answer(
    query='Explain overfitting and how regularization techniques like L1 and L2 help.',
    k=5
)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3596.76ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2841.36ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6823.38ms



══════════════════════════════════════════════════════════════════════
❓ QUESTION: Explain overfitting and how regularization techniques like L1 and L2 help.
──────────────────────────────────────────────────────────────────────
🤖 ANSWER (Gemini 2.5 Flash | k=5 | pages [63, 66, 72, 76, 81]):

Overfitting occurs when a model performs very well on the training
   data but poorly on unseen test data. This discrepancy between
   training and test set performance is a clear sign of overfitting
   (page 63
══════════════════════════════════════════════════════════════════════


In [31]:
# Query 3
r3 = rag_answer(
    query='How do convolutional neural networks (CNNs) process image data?',
    k=5
)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3852.46ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6866.82ms



══════════════════════════════════════════════════════════════════════
❓ QUESTION: How do convolutional neural networks (CNNs) process image data?
──────────────────────────────────────────────────────────────────────
🤖 ANSWER (Gemini 2.5 Flash | k=5 | pages [16, 44, 154, 319, 379]):

The provided excerpts do not contain enough information to answer this
   question fully.
══════════════════════════════════════════════════════════════════════


In [32]:
# Query 4
r4 = rag_answer(
    query='What are the main differences between bagging and boosting ensemble methods?',
    k=7
)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3018.73ms



══════════════════════════════════════════════════════════════════════
❓ QUESTION: What are the main differences between bagging and boosting ensemble methods?
──────────────────────────────────────────────────────────────────────
🤖 ANSWER (Gemini 2.5 Flash | k=7 | pages [44, 97, 98, 103, 346, 381, 383]):

Ensemble methods combine multiple machine learning models to create
   more powerful models (Page 97). Two prominent ensemble methods that
   use decision trees as building blocks are random forests and
   gradient boosted decision trees
══════════════════════════════════════════════════════════════════════
